# How old is the universe?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F06_age_of_the_universe.ipynb).

Every galaxy far enough away from us is moving away, and the further away it is the faster it
goes. That one observation is the most consequential measurement in astronomy, because if
everything is flying apart now then everything was in the same place once — and the speed tells
you how long ago.

Today you get thirty-six of those measurements. Each one is a Type Ia supernova: an exploding
star bright enough to be seen most of the way across the universe, and regular enough that its
brightness gives away its distance. Distance on one axis, speed on the other, and the slope of
the line through them has a length of time hidden inside it.

Getting that time right turns on a single decision, and it is not a statistical decision — it is
a physical one. You will make the same decision a second time before the end of the notebook, on
the floor of the Pacific Ocean.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, then export the notebook as a PDF and upload that.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say how old the universe is, in billions of years, and say exactly which
measurement that number came from. Say how fast two ocean ridges are building new seafloor, and
why one of them is several times faster than the other.

**The skills.** Fit a straight line with scikit-learn: `LinearRegression().fit(x, y)`, then
`.coef_` for the slope, `.intercept_` for where it crosses zero, `.predict()` for what the line
says, and `.score()` for how much of the data it accounts for. Subtract the line from the data to
get **residuals**, and look at them. And fit a line that is not allowed an intercept at all,
`LinearRegression(fit_intercept=False)`, when physics has already told you one point it must pass
through.

**Eight places where you write something: five in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it.

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(archive_path, cached_name):
    """Read one of this week's tables from the archive it came from; fall back to the copy stored with the course."""
    try:
        return pd.read_csv(f"https://raw.githubusercontent.com/AI4EPS/EPS88_2024/a58436d0/{archive_path}")
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + cached_name)

STARS = "Week08_Age_of_the_Universe/Data/"
RIDGES = "Week05_Seafloor_Spreading/data/"

sn = load(STARS + "Freedman2000_Supernova1a.csv", "week06_supernovae.csv")
par = load(RIDGES + "PAR_east_age_dist.csv", "week06_par_age_distance.csv")
mar = load(RIDGES + "MAR_east_age_dist.csv", "week06_mar_age_distance.csv")
coast = pd.read_csv(CACHE + "/coastlines.csv")

print("supernovae:", sn.shape, " Pacific-Antarctic:", par.shape, " Mid-Atlantic:", mar.shape)

## Thirty-six exploding stars

The table is small enough to read. It is the Type Ia supernova table from the *Hubble Space
Telescope* Key Project (Freedman et al., *The Astrophysical Journal* **553**, 47, 2001), the study
that set out to pin the expansion rate down, and it has been handed round this course since 2019.

In [ ]:
print("rows:", len(sn))
print("distance:", sn["D(Mpc)"].min(), "to", sn["D(Mpc)"].max(), "Mpc")
print("speed:   ", sn["VCMB"].min(), "to", sn["VCMB"].max(), "km/s")
sn.head()

Four numbers matter, and one of them is a column name you have to type in quotes because it has
brackets in it.

- `Supernova` — which exploding star this row is.
- `VCMB` — how fast that supernova's galaxy is moving away from us, in km/s. `CMB` says the speed
  has been corrected for the Sun's own motion, so it is a speed relative to the universe rather
  than to us.
- `D(Mpc)` — how far away it is, in **megaparsecs**. A parsec is 3.0857 × 10¹³ km, about 3.26
  light years (the International Astronomical Union's definition of the parsec, read 2026-08-31),
  and a megaparsec is a million of them.
- `HCMB` — the speed divided by the distance, which the paper printed for every supernova.

So the nearest of these supernovae is 56 Mpc away and the farthest
467 Mpc — eight times further — and every one of them is receding from us. Plot the
two columns against each other and the shape of the relationship is immediate.

In [ ]:
plt.scatter(sn["D(Mpc)"], sn["VCMB"])
plt.xlabel("distance (Mpc)")
plt.ylabel("recession speed (km/s)")
plt.title("Type Ia supernovae, 36 of them")
plt.show()

Further away means faster, and close enough to a straight line that you could put a ruler on it.
That is **Hubble's law**: speed = H₀ × distance, where H₀ is the number we are after. It is
measured in the awkward-looking units of kilometres per second per megaparsec — how much faster a
galaxy recedes for every extra megaparsec of distance.

Notice what those units are hiding. Kilometres per second divided by megaparsecs is a speed
divided by a distance, and a speed divided by a distance is one over a time. So H₀ has a time
buried in it, and that is the whole trick of this notebook.

Before any line-fitting, the crudest possible estimate. Every single supernova already gives you
its own H₀ — just divide.

### ✏️ Your turn 1

Divide the speed column by the distance column, all thirty-six at once, and see how much the
answers disagree. Print the smallest, the largest, the average, and the largest divided by the
smallest.

Then check yourself against the paper: the `HCMB` column is the same division, done by the
authors. Print its average too, and the two should match.

**Use these names**, because the self-check looks for them: `h_each`.

In [ ]:
# ← your answer here


assert len(h_each) == 36, "h_each should hold one number per supernova, not one number"
print("✓ one Hubble constant per supernova —", round(h_each.min(), 1), "to",
      round(h_each.max(), 1), "km/s/Mpc, which is not one answer but",
      len(h_each), "of them")

Thirty-six supernovae, 64.8 to 83.7 km/s/Mpc, the largest
1.29 times the smallest. That is not one Hubble constant, it is
thirty-six of them — and since H₀ has a time inside it, thirty-six different ages for the
universe, the longest 1.29 times the shortest. Dividing one supernova by
itself throws away the other thirty-five, and every supernova carries measurement error in both
its speed and its distance.

What we want is one number that uses all thirty-six at once.

## One line through all of them

Draw the best straight line. Best means the smallest total miss. For each point, the miss is the
vertical gap between the real speed and the speed the line claims; square each gap so that
misses above and below cannot cancel; add them up; and choose the line that makes that total as
small as it can be. There is exactly one such line, and scikit-learn will find it.

```
model = LinearRegression().fit(x, y)
```

Two details in that one line. `x` has to be a **table of columns**, not a single column, because
in later weeks you will fit on several columns at once — so it is `sn[["D(Mpc)"]]` with two sets
of brackets, meaning "a table containing this one column", while `y` stays a single column,
`sn["VCMB"]`. And `fit` hands you back the model, which then answers questions: `.coef_[0]` is
the slope, `.intercept_` is where the line crosses x = 0, `.predict()` is what the line says at
any distance, and `.score()` is R², the fraction of the up-and-down variation in the data that
the line accounts for — 1.0 would be every point exactly on the line.

### Predict before you run

The line is about to tell you what speed it expects at a distance of **zero** — a galaxy right
here, no distance away at all. What number should that be? Commit to it in the next cell before
you run it.

In [ ]:
my_guess_intercept = 0        # ← change this to whatever you think, then run the cell

model = LinearRegression().fit(sn[["D(Mpc)"]], sn["VCMB"])

print("you guessed: ", my_guess_intercept, "km/s")
print("slope:       ", round(model.coef_[0], 2), "km/s per Mpc")
print("intercept:   ", round(model.intercept_, 1), "km/s")
print("R²:          ", round(model.score(sn[["D(Mpc)"]], sn["VCMB"]), 4))

An R² of 0.978 says the line accounts for almost all of the spread in the speeds,
and a slope of 67.5 km/s/Mpc is in the right neighbourhood of the
supernova-by-supernova numbers you printed. The intercept is the surprise, and we will come back
to it in a moment.

Draw the line on top of the points first. `.predict()` takes the same table of distances and
returns what the line says at each one.

In [ ]:
predicted = model.predict(sn[["D(Mpc)"]])

plt.scatter(sn["D(Mpc)"], sn["VCMB"], label="supernovae")
plt.plot(sn["D(Mpc)"], predicted, color="black", label="best straight line")
plt.xlabel("distance (Mpc)")
plt.ylabel("recession speed (km/s)")
plt.title("Type Ia supernovae and the fitted line, 36 points")
plt.legend()
plt.show()

The line looks right, but "looks right" is not a measurement. The honest way to see what a fit is
doing wrong is to subtract it: for each point, the **residual** is the real value minus the value
the line predicts. A residual is what the line failed to explain, and plotting the residuals
against distance is a much harsher test than plotting the fit — the line is now flat, at zero, so
any pattern left in the picture is a pattern the line missed.

### ✏️ Your turn 2

Compute the residuals — the real speeds minus `predicted` — and plot them against distance, with
a horizontal line at zero to fit against. Then print the biggest miss in each direction, using
`.max()` and `.min()`.

Look at the picture before you move on: are the misses scattered evenly above and below zero
across the whole range of distances, or do they drift systematically to one side at one end?

**Use these names**, because the self-check looks for them: `residuals`.

In [ ]:
# ← your answer here


assert abs(residuals.mean()) < 1, \
    "these should be the misses of the free fit, whose residuals always average to zero"
print("✓ residuals — the worst miss is", round(residuals.abs().max()),
      "km/s, on a fit that averages", round(residuals.mean(), 6), "km/s off")

The worst miss is about 3,169 km/s above the line and 1,998
km/s below it, and both signs turn up at every distance rather than the picture sweeping from one
side of zero to the other as you move right. So the straight line is a fair description of these
data, and a curve would not obviously do better. That settles the *shape*. It does not settle the
*position*.

## Where the line crosses zero

Go back to the intercept, which is the number your prediction was about. The fitted line says
that at a distance of zero — right here, no distance at all — a galaxy is already receding at
712 km/s.

### ✏️ Your turn 3

In two or three sentences: what would it actually mean for a galaxy at zero distance to be moving
away from us at that speed, and is that something the universe does? Say what you think should be
done about it.

*(This one is written, not coded — answer in the cell below.)*

*(Double-click this cell and replace this line with your answer.)*

Sometimes physics already knows one point on the line. Make the line go through it. In
scikit-learn that is one extra argument, and nothing else about the fit changes:

```
model = LinearRegression(fit_intercept=False).fit(x, y)
```

`fit_intercept=False` says: do not look for a best crossing point, there isn't one to look for —
the line goes through (0, 0) and the only thing left to choose is its slope.

And now the slope is the whole answer, because of those units. H₀ is a speed over a distance,
which is one over a time, so **1 / H₀ is a time**: how long ago everything was in the same place,
if galaxies have always moved at the speed they are moving now. Two conversions turn it into
years — the megaparsec in kilometres from the definition above, and a year of 365.25 days, which
is the astronomers' convention rather than a measurement.

### ✏️ Your turn 4

Fit the supernovae again, this time with `fit_intercept=False`, and print the slope: that is our
measured H₀.

Then turn it into an age. Write a function `age_of_universe(H0)` with a docstring, which takes a
Hubble constant in km/s per Mpc and returns an age in **billions of years**. Inside it:

```
MPC_IN_KM = 3.0857e19             # one megaparsec, in kilometres
SECONDS_PER_YEAR = 60 * 60 * 24 * 365.25
```

1 / H₀ is in Mpc·s/km, so multiplying by `MPC_IN_KM` gives seconds; divide by
`SECONDS_PER_YEAR` for years, and by `1e9` for billions of years.

Call it twice and print both answers: once on your H₀, and once on the slope of the free fit from
earlier, `model.coef_[0]`. The second one is what the constraint cost you if you had left it out.

**Use these names**, because the self-check looks for them: `H0`, `age_of_universe`, `age_Gyr`.

In [ ]:
# ← your answer here


assert 1 < age_Gyr < 100, \
    "age_Gyr should be a number of BILLIONS of years — check the last two divisions"
print("✓ the age of the universe —", round(age_Gyr, 2), "billion years, from a slope of",
      round(H0, 2), "km/s/Mpc")

Two checks on that, and both are external.

The paper these data come from quotes 71 ± 2 (random) ± 6 (systematic) km/s/Mpc from
its Type Ia supernovae; you got 70.67 from the same table. And the current
best measurement of the age of the universe, made a completely different way — from the Planck
satellite's map of the microwave background — is 13.797 ± 0.023 billion years (Planck
Collaboration, *Astronomy & Astrophysics* **641**, A6, 2020; both figures read 2026-08-31). Your
13.84 sits 0.04 billion years above it.

That is a startlingly good answer for an hour's work, and it deserves one honest caveat. 1 / H₀ is
the age only if the expansion has always run at today's rate. It has not: gravity slowed it down
early on, and over the last few billion years it has been speeding up again. Those two effects
very nearly cancel in our universe, which is why the simple answer lands so close to the careful
one. And notice what the constraint bought. The free fit, with its 712
km/s intercept, gives 14.48 billion years — 0.68 billion
years out, for a reason that has nothing to do with cosmology and everything to do with a line
nobody told where to start.

## The same question on the ocean floor

Nothing in the last three sections was about astronomy. Two columns, a straight line, a slope with
a time in it, and one physical constraint on where the line has to pass. That pattern is
everywhere, and the nearest example is under the sea.

New ocean floor is made at mid-ocean ridges: two plates pull apart, molten rock rises into the
gap and freezes onto both edges. As it freezes it records the direction of Earth's magnetic field,
which flips every so often, so the seafloor carries a barcode of magnetic stripes running parallel
to the ridge — and each stripe can be dated, because the pattern of flips is known. Measure how
far a dated stripe now sits from the ridge that made it and you have exactly the shape of problem
you just solved: a distance, an age, and a slope that is a speed.

The two tables are picks of those stripes along ship tracks, one set east of the
**Pacific-Antarctic Ridge** in the far South Pacific and one east of the **Mid-Atlantic Ridge**.
They came into the course with the earlier offerings and carry no source note of their own, which
is worth knowing about any file you did not make. Each row is one pick: its age in millions of
years, where it is, and its distance in kilometres. (The first column is the pick number the
original files were saved with; ignore it.)

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
par = load(RIDGES + "PAR_east_age_dist.csv", "week06_par_age_distance.csv")
mar = load(RIDGES + "MAR_east_age_dist.csv", "week06_mar_age_distance.csv")

In [ ]:
par.head()

Where those picks are matters, so put them on the map before fitting anything. Longitude and
latitude are just two more columns, drawn as an ordinary scatter over the coastline.

In [ ]:
plt.figure(figsize=(9, 5))          # a world map needs to be wider than the house default
plt.plot(coast["lon"], coast["lat"], color="0.6", lw=0.6)
plt.scatter(par["Lon"], par["Lat"], s=6, label="Pacific-Antarctic Ridge")
plt.scatter(mar["Lon"], mar["Lat"], s=6, label="Mid-Atlantic Ridge")
plt.xlim(-180, 180)
plt.ylim(-90, 90)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title("Magnetic-stripe picks on two ridges, 375 of them")
plt.legend(loc="lower left")
plt.show()

Two lines of dots, each trailing away from its ridge: the Pacific-Antarctic picks run southeast
from about 55° S, the Mid-Atlantic picks run west to east across the Atlantic at about 25° N.
Neither is near a coast, which is the point — this is seafloor, not continent.

Now the same plot as the supernovae, with age where distance was.

In [ ]:
plt.scatter(par["Age"], par["Distance"], s=10)
plt.xlabel("age of the seafloor (millions of years)")
plt.ylabel("distance from the ridge (km)")
plt.title("Pacific-Antarctic Ridge, 140 picks")
plt.show()

Straight, and the physical constraint is even more obvious here than it was for the supernovae:
seafloor of age zero is being made at the ridge right now, so it is zero kilometres from the
ridge. The line has a point it must pass through, and it is the origin again.

### ✏️ Your turn 5

Fit the Pacific-Antarctic picks **both ways** — once free, once with `fit_intercept=False` — with
`par[["Age"]]` as x and `par["Distance"]` as y.

Print, for the free fit, its slope and its intercept; and for the forced fit, its slope. Report
the slopes in **centimetres per year** as well as km per million year: one km per million years is
0.1 cm per year, so divide by 10.

Then draw both lines on the scatter, so you can see how far apart they are.

**Use these names**, because the self-check looks for them: `par_free`, `par_forced`.

In [ ]:
# ← your answer here


assert par_forced.intercept_ == 0, \
    "the forced model should have no intercept at all — did you pass fit_intercept=False?"
print("✓ the Pacific-Antarctic Ridge — free fit",
      round(par_free.coef_[0] / 10, 2), "cm/yr with the ridge",
      round(par_free.intercept_), "km from itself; forced through the origin",
      round(par_forced.coef_[0] / 10, 2), "cm/yr")

The free fit puts the ridge 181 km away from the ridge at age zero.
There is no reading of that which is physically possible: 181 km of
seafloor cannot exist before any seafloor has been made. It is the 712
km/s intercept again, in different clothes, and it comes from the same cause — the youngest pick
in the file is 0.78 million years old, so the fit is extrapolating past the end of
its own data. Forcing the line through the origin moves the answer from
6.11 to 6.77 cm/yr —
0.66 cm/yr of difference produced by an
argument rather than by any new data.

## A second ridge

One ridge is not a result. The Mid-Atlantic table has 235 picks, and the fit is lines you
have already written.

In [ ]:
mar_forced = LinearRegression(fit_intercept=False).fit(mar[["Age"]], mar["Distance"])

print("Mid-Atlantic:      ", round(mar_forced.coef_[0] / 10, 2), "cm/yr, R²",
      round(mar_forced.score(mar[["Age"]], mar["Distance"]), 3))
print("Pacific-Antarctic: ", round(par_forced.coef_[0] / 10, 2), "cm/yr, R²",
      round(par_forced.score(par[["Age"]], par["Distance"]), 3))
print("ratio:             ", round(par_forced.coef_[0] / mar_forced.coef_[0], 2))

In [ ]:
plt.scatter(mar["Age"], mar["Distance"], s=10, label="Mid-Atlantic picks")
plt.scatter(par["Age"], par["Distance"], s=10, label="Pacific-Antarctic picks")
plt.plot(mar["Age"], mar_forced.predict(mar[["Age"]]), color="black")
plt.plot(par["Age"], par_forced.predict(par[["Age"]]), color="red")
plt.xlabel("age of the seafloor (millions of years)")
plt.ylabel("distance from the ridge (km)")
plt.title("Two ridges, 375 picks, both forced through the origin")
plt.legend()
plt.show()

The Mid-Atlantic line rises at 1.74 cm/yr and the Pacific-Antarctic
one at 6.77 cm/yr — 3.9 times as steep, over
83 million years of Atlantic seafloor and 41 million years
of Pacific. Both are roughly the speed a fingernail grows, and both fits account for over
98 % of the variation in the distances, so
the gap between them is not slop in the fitting.

It is not slop in the Earth either. Plates are pulled along mainly by their own sinking edges:
where old, cold ocean floor bends down into the mantle at a trench, its weight drags the rest of
the plate after it, and that force dominates everything else (Forsyth & Uyeda, *Geophysical
Journal of the Royal Astronomical Society* **43**, 163, 1975). The Pacific plate is ringed almost
all the way round by trenches. The plates on either side of the Mid-Atlantic Ridge — North America
and Africa — have almost no sinking edge anywhere. So the Atlantic opens slowly and the Pacific
opens fast, and the number you fitted is a measurement of that difference.

## The question, answered

About 13.84 billion years — the slope of a straight line through
36 exploding stars, forced through the origin because zero distance has to mean zero
recession, and landing 0.04 billion years from the
13.797 ± 0.023 billion years the microwave background gives.

## Week 6 summary

**The question.** How old is the universe?

### What to remember

| | |
|---|---|
| **1** | A slope can be an age. |
| **2** | Physics can force a fit through the origin; the free fit was not wrong, it was unconstrained. |
| **3** | Two ocean ridges spread at rates differing fourfold, and that difference is the science. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Linear regression** | Draw the best straight line. Best means the smallest total miss. |
| **Fit through the origin** | Sometimes physics already knows one point on the line. Make the line go through it. |

## Homework

Three parts on the two datasets you already have loaded. Parts 1 and 2 are two versions of the
same worry: class fitted one line to a whole table and read one number off it, and neither table
was asked whether a single line is really enough. Part 3 is where you argue from your own numbers.
If you have restarted since class, run the setup cell at the top, then the cell below, and then
your own answer to *Your turn 4*, so that `age_of_universe` exists again.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
sn = load(STARS + "Freedman2000_Supernova1a.csv", "week06_supernovae.csv")
par = load(RIDGES + "PAR_east_age_dist.csv", "week06_par_age_distance.csv")

### ✏️ Your turn 6

If Hubble's law is really a law, then the 36 supernovae should give the same H₀ whichever
ones you use. Test it on the two halves of the table.

Sort the table by distance with `sn.sort_values("D(Mpc)")`, take the nearest 18 with `.head(18)`
and the farthest 18 with `.tail(18)`, and fit each half through the origin. Print both Hubble
constants, and put both through your `age_of_universe` function from class to get two ages.

**Use these names**, because the self-check looks for them: `H0_near`, `H0_far`, `age_near`,
`age_far`.

In [ ]:
# ← your answer here


assert H0_near != H0_far, \
    "two different halves of the table cannot give the identical fit — check you split it"
print("✓ near against far — the two halves differ by",
      round(abs(age_near - age_far), 2), "billion years")

### ✏️ Your turn 7

Now the same worry about the Pacific-Antarctic Ridge, where it has more bite. Class fitted one
line to all 140 picks and read off a single rate — but the picks run from
0.78 to 41 million years, and there is no law saying a ridge
must keep the same speed for forty million years.

Take only the older picks, `old = par[par["Age"] >= 20]`, and fit **those** both ways: free, and
forced through the origin. Print both slopes in cm/yr and the free fit's intercept. Then draw the
picks you kept with both of your lines on top.

This is the fork, and this time the two answers are both defensible. Forcing through the origin
says *the ridge existed at age zero, so the line must start there*. Fitting free says *I am asking
how fast this ridge moved between 20 and 41 million years ago, and the
origin is outside that window*. Part 3 is where you choose.

**Use these names**, because the self-check looks for them: `old`, `old_free`, `old_forced`.

In [ ]:
# ← your answer here


assert old["Age"].min() >= 20, "old should keep the picks 20 Ma and OLDER"
print("✓ the older half of the ridge — free", round(old_free.coef_[0] / 10, 2),
      "cm/yr against forced", round(old_forced.coef_[0] / 10, 2), "cm/yr")

### ✏️ Your turn 8

Four numbers, all yours: the whole-ridge rate from class (6.77 cm/yr,
forced through the origin), the Mid-Atlantic rate from class
(1.74 cm/yr), and your two rates for the older half of the
Pacific-Antarctic picks.

Quote all four, then answer both of these in a short paragraph.

**Which of your two fits for the older half would you publish** as the rate for the last part of
that window, and why? Your answer has to deal with the free fit's intercept, which is no longer
close to zero.

**And is the difference *between* the two ridges bigger or smaller than the difference *within*
the Pacific-Antarctic Ridge?** Give both as a ratio, from your own numbers, and say what your
answer implies about whether "the spreading rate of a ridge" is one number or several.

*(Double-click this cell and replace this line with your answer.)*